# 04 — Frozen-residual anomaly models

This notebook implements one clear sequence:

1. measurement-kind transformations;
2. a calibration-frozen robust reference;
3. rapid, drift and dispersion statistical channels;
4. optional PCA and Isolation Forest on the same residuals;
5. comparison at equal **case** workload on development data.

The holdout is not opened here.


## 1. Setup


In [ ]:
import importlib
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT = Path(
    os.getenv("ANOMALY_DATA_ROOT")
    or os.getenv("ANOMALY_DRIVE_ROOT")
    or ("/content/drive/MyDrive/anomaly_detection" if IN_COLAB
        else Path.home() / "anomaly_detection_data")
).expanduser()
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DATA_ROOT / "research" / "milestone1" if IN_COLAB
    else Path.cwd() if (Path.cwd() / "milestone1_core.py").is_file()
    else Path.cwd() / "notebooks" / "drive_research",
)).expanduser()
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

SECTOR = os.getenv("ANOMALY_SECTOR", "telecom")  # or "petrobras_3w"
CANONICAL_RUN_IDS = {
    "telecom": "telecom_core_v0_10_1_run1",
    "petrobras_3w": "petrobras_3w_core_v0_10_1_run2",
}
if SECTOR not in CANONICAL_RUN_IDS:
    raise ValueError(f"Choose one of {list(CANONICAL_RUN_IDS)}")

from milestone1_core import CORE_VERSION, new_output_directory, read_json, write_json

CANONICAL_RUN_ID = os.getenv("CANONICAL_RUN_ID", CANONICAL_RUN_IDS[SECTOR])
RUN_ROOT = DATA_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}" / SECTOR / CANONICAL_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
EVAL_ROOT = RUN_ROOT / "SPEC-EVAL"
SPLIT_ROOT = RUN_ROOT / "SPLITS"

import tempfile
import time

import duckdb
import joblib

import evaluation_core
import simple_model_core
evaluation_core = importlib.reload(evaluation_core)
simple_model_core = importlib.reload(simple_model_core)
from evaluation_core import evaluate_cases, form_cases
from simple_model_core import (
    MODEL_CORE_VERSION, MODEL_IDS, alert_grid_from_score_file,
    calibration_thresholds, fit_residual_bundle,
    materialize_measurement_features, materialize_wide_partition,
    partition_exposure, score_residual_file,
)

EDA_VERSION = EVALUATION_VERSION = MODEL_VERSION = "2.0.0"
EDA_ROOT = DATA_ROOT / "outputs" / "eda" / f"v{EDA_VERSION}" / SECTOR / f"{SECTOR}_eda_v2_run1"
EVALUATION_ROOT = DATA_ROOT / "outputs" / "evaluation" / f"v{EVALUATION_VERSION}" / SECTOR / f"{SECTOR}_evaluation_v2_run1"
MODEL_ROOT = DATA_ROOT / "outputs" / "models" / f"v{MODEL_VERSION}" / SECTOR / f"{SECTOR}_models_v2_run1"
MAX_TRAINING_ROWS = int(os.getenv("MODEL_MAX_TRAINING_ROWS", "150000"))
ISOLATION_TREES = int(os.getenv("ISOLATION_TREES", "200"))

manifest = read_json(CORE_ROOT / "manifest.json")
decisions = read_json(EDA_ROOT / "eda_decisions.json")
policy = read_json(EVALUATION_ROOT / "evaluation_policy.json")
catalogue = pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet")
fault_events = pd.read_parquet(EVALUATION_ROOT / "development" / "fault_events.parquet")
fault_intervals = pd.read_parquet(EVALUATION_ROOT / "development" / "fault_entity_intervals.parquet")
group_path = SPLIT_ROOT / "entity_groups.parquet"
entity_groups = pd.read_parquet(group_path) if group_path.is_file() else pd.DataFrame()

if decisions["canonical_fingerprint"] != manifest["fingerprint"]:
    raise ValueError("EDA and canonical input do not match")
if policy["canonical_fingerprint"] != manifest["fingerprint"]:
    raise ValueError("Evaluation and canonical input do not match")

display(pd.Series({
    "sector": SECTOR,
    "reference": "frozen calibration median / robust scale",
    "training_rows_cap": MAX_TRAINING_ROWS,
    "isolation_trees": ISOLATION_TREES,
    "output": str(MODEL_ROOT),
}, name="value").to_frame())


## 2. Transform, fit on calibration and score development


In [ ]:
started = time.perf_counter()
temporary = tempfile.TemporaryDirectory()
work = Path(temporary.name)
paths = {}

for partition in ("calibration", "development"):
    wide = work / f"{partition}_wide.parquet"
    feature_path = work / f"{partition}_features.parquet"
    split = materialize_wide_partition(
        CORE_ROOT, SPLIT_ROOT, partition, catalogue, wide,
        lookback_seconds=decisions["base_cadence_seconds"],
    )
    materialize_measurement_features(
        wide, catalogue, feature_path,
        score_start=split["score_start"], score_end=split["score_end"],
    )
    paths[partition] = {"wide": wide, "features": feature_path, **split}

bundle = fit_residual_bundle(
    paths["calibration"]["features"],
    use_entity_reference=decisions["primary_split"] == "time",
    maximum_training_rows=MAX_TRAINING_ROWS,
    isolation_trees=ISOLATION_TREES,
)
for partition in paths:
    score_path = work / f"{partition}_scores.parquet"
    score_residual_file(
        bundle, paths[partition]["features"], score_path,
        cadence_seconds=decisions["base_cadence_seconds"],
        dispersion_window_seconds=decisions["dispersion_window_seconds"],
    )
    paths[partition]["scores"] = score_path

display(pd.Series({
    "fitted_features": len(bundle["feature_columns"]),
    "entity_specific_reference": bundle["use_entity_reference"],
    "elapsed_minutes": (time.perf_counter() - started) / 60,
}, name="value").to_frame())


## 3. Calibration thresholds and readiness


In [ ]:
block_column = "entity_id" if decisions["primary_split"] == "time" else "episode_id"
thresholds = calibration_thresholds(
    paths["calibration"]["scores"],
    policy["threshold_quantiles"],
    block_column=block_column,
    minimum_block_rows=20,
    model_ids=MODEL_IDS,
)

with duckdb.connect() as connection:
    score_file = str(paths["development"]["scores"])
    readiness = connection.execute("""
        SELECT entity_id, readiness, count(*) AS observations
        FROM read_parquet(?)
        GROUP BY entity_id, readiness
        ORDER BY entity_id, readiness
    """, [score_file]).df()
display(thresholds)
display(readiness.groupby("readiness").observations.sum().to_frame())


## 4. Compare three predeclared portfolios

The sweep is intentionally small. Every channel is calibrated on its own
calibration distribution, then alerts are consolidated into cases before
workload and recall are measured.


In [ ]:
PORTFOLIOS = {
    "rapid_only": ["rapid_residual"],
    "statistical_v1": ["rapid_residual", "drift_cusum", "dispersion_change"],
    "residual_multivariate": list(MODEL_IDS),
}
PREFERENCE = {name: rank for rank, name in enumerate(PORTFOLIOS)}
exposure = partition_exposure(
    paths["development"]["scores"], policy["exposure_unit"],
    decisions["base_cadence_seconds"],
)

def threshold_map(quantile, channels):
    selected = thresholds.loc[
        thresholds.threshold_quantile.eq(quantile)
        & thresholds.model_id.isin(channels)
    ]
    return dict(zip(selected.model_id, selected.threshold))

alert_grid = alert_grid_from_score_file(
    paths["development"]["scores"], thresholds,
    persistence=policy["channel_persistence"],
    recovery_consecutive=policy["recovery_observations"],
)

def make_alerts(channels, quantile):
    channel_thresholds = threshold_map(quantile, channels)
    frames = [alert_grid[(channel, float(quantile))] for channel in channels]
    frames = [frame for frame in frames if not frame.empty]
    if not frames:
        from evaluation_core import ALERT_COLUMNS
        return pd.DataFrame(columns=ALERT_COLUMNS), channel_thresholds
    alerts = pd.concat(frames, ignore_index=True).sort_values("alert_start").reset_index(drop=True)
    alerts["alert_id"] = [f"A-{number:09d}" for number in range(1, len(alerts) + 1)]
    return alerts, channel_thresholds

comparisons = []
candidate_cache = {}
for portfolio, channels in PORTFOLIOS.items():
    for quantile in policy["threshold_quantiles"]:
        alerts, channel_thresholds = make_alerts(channels, quantile)
        cases, members = form_cases(
            alerts, entity_groups,
            gap_seconds=policy["case_gap_seconds"],
            thresholds=channel_thresholds,
        )
        result = evaluate_cases(
            cases, members, fault_events, fault_intervals,
            exposure_value=exposure,
            exposure_unit=policy["exposure_unit"],
            decision_horizon_seconds=policy["decision_horizon_seconds"],
        )
        metrics = result["metrics"].set_index("metric").value
        key = f"{portfolio}|{quantile}"
        candidate_cache[key] = (alerts, cases, members, result, channel_thresholds)
        comparisons.append({
            "candidate_key": key,
            "portfolio": portfolio,
            "threshold_quantile": quantile,
            "channels": ", ".join(channels),
            "event_recall": metrics["event_recall"],
            "preimpact_event_recall": metrics["preimpact_event_recall"],
            "case_precision": metrics["case_precision"],
            "false_case_rate": metrics[f"false_cases_per_{policy['exposure_unit']}"],
            "total_case_rate": metrics[f"total_cases_per_{policy['exposure_unit']}"],
            "median_delay_seconds": metrics["median_detection_delay_seconds"],
            "cases": len(cases),
        })
        print(f"Evaluated {portfolio}, q={quantile}: {len(cases):,} cases")

comparison = pd.DataFrame(comparisons)
display(comparison.sort_values(["portfolio", "threshold_quantile"]))


## 5. Apply the predeclared selection rule


In [ ]:
eligible = comparison.loc[comparison.false_case_rate.le(policy["false_case_budget"])].copy()
selection_status = "within_budget"
if eligible.empty:
    eligible = comparison.nsmallest(1, "false_case_rate").copy()
    selection_status = "no_configuration_within_budget"

best_recall = eligible.event_recall.max()
equivalent = eligible.loc[eligible.event_recall.ge(best_recall - 0.02)].copy()
equivalent["preference"] = equivalent.portfolio.map(PREFERENCE)
selected = equivalent.sort_values(
    ["preference", "false_case_rate", "median_delay_seconds", "threshold_quantile"],
    ascending=[True, True, True, False],
).iloc[0]

alerts, cases, members, result, selected_thresholds = candidate_cache[selected.candidate_key]
configuration = {
    "model_version": MODEL_VERSION,
    "model_core_version": MODEL_CORE_VERSION,
    "sector": SECTOR,
    "canonical_fingerprint": manifest["fingerprint"],
    "portfolio": selected.portfolio,
    "channels": PORTFOLIOS[selected.portfolio],
    "threshold_quantile": float(selected.threshold_quantile),
    "thresholds": selected_thresholds,
    "selection_status": selection_status,
    "false_case_budget": policy["false_case_budget"],
    "development_exposure": exposure,
    "exposure_unit": policy["exposure_unit"],
    "case_gap_seconds": policy["case_gap_seconds"],
    "channel_persistence": policy["channel_persistence"],
    "recovery_observations": policy["recovery_observations"],
    "decision_horizon_seconds": policy["decision_horizon_seconds"],
    "feature_settings": decisions,
    "holdout_used": False,
}

display(pd.Series(configuration, name="development_reference").to_frame())
if selection_status != "within_budget":
    print("STOP — no configuration met the development workload gate; holdout must remain sealed")
display(result["metrics"])
display(result["fault_type_results"])


## 6. Persist the small development evidence set


In [ ]:
cases = cases.sort_values("anomaly_evidence_score", ascending=False).reset_index(drop=True)
cases.insert(0, "rank", np.arange(1, len(cases) + 1))

with new_output_directory(MODEL_ROOT) as output:
    joblib.dump(bundle, output / "residual_bundle.joblib")
    write_json(output / "selected_configuration.json", configuration)
    thresholds.to_csv(output / "calibration_thresholds.csv", index=False)
    comparison.to_csv(output / "development_comparison.csv", index=False)
    readiness.to_parquet(output / "development_readiness.parquet", index=False)
    alerts.to_parquet(output / "selected_alerts.parquet", index=False)
    cases.to_parquet(output / "selected_cases.parquet", index=False)
    members.to_parquet(output / "selected_case_members.parquet", index=False)
    result["metrics"].to_csv(output / "development_metrics.csv", index=False)
    result["fault_type_results"].to_csv(output / "fault_type_results.csv", index=False)

print("Saved:", MODEL_ROOT)
print("Development reference portfolio:", configuration["portfolio"])
print("Next: 05_INCIDENT_RANKING_AND_HOLDOUT.ipynb")
temporary.cleanup()
